In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
import os

DATA = "../data/processed/"
CHARTS = "../reports/eda_charts/"
os.makedirs(CHARTS, exist_ok=True)

nav_history = pd.read_csv(DATA + "nav_history_clean.csv")
fund_master = pd.read_csv(DATA + "fund_master_clean.csv")
benchmark = pd.read_csv(DATA + "benchmark_indices_clean.csv")

nav_history["date"] = pd.to_datetime(nav_history["date"])
nav_history = nav_history.sort_values(["amfi_code", "date"])

In [22]:
nav_history["daily_return"] = nav_history.groupby("amfi_code")["nav"].pct_change()

print(nav_history["daily_return"].describe())
# Validate distribution looks reasonable — most daily equity fund returns should be small, roughly -5% to +5%
print("Extreme outliers (>10% daily move):", (nav_history["daily_return"].abs() > 0.10).sum())

count    64280.000000
mean         0.000451
std          0.008706
min         -0.058102
25%         -0.002092
50%          0.000000
75%          0.003233
max          0.064713
Name: daily_return, dtype: float64
Extreme outliers (>10% daily move): 0


In [23]:
def calc_cagr(nav_start, nav_end, years):
    if nav_start <= 0 or years <= 0:
        return np.nan
    return (nav_end / nav_start) ** (1/years) - 1

cagr_results = []
latest_date = nav_history["date"].max()

for code, group in nav_history.groupby("amfi_code"):
    group = group.set_index("date")
    row = {"amfi_code": code}
    for years, label in [(1, "cagr_1yr"), (3, "cagr_3yr"), (5, "cagr_5yr")]:
        target_date = latest_date - pd.DateOffset(years=years)
        past = group[group.index <= target_date]
        if len(past) > 0 and len(group) > 0:
            nav_start = past["nav"].iloc[-1]
            nav_end = group["nav"].iloc[-1]
            actual_years = (group.index[-1] - past.index[-1]).days / 365.25
            row[label] = calc_cagr(nav_start, nav_end, actual_years)
        else:
            row[label] = np.nan
    cagr_results.append(row)

cagr_df = pd.DataFrame(cagr_results)
cagr_df = cagr_df.merge(fund_master[["amfi_code","scheme_name"]], on="amfi_code")
cagr_df

,amfi_code,cagr_1yr,cagr_3yr,cagr_5yr,scheme_name
0,100016,-0.022258,0.012924,NaN,HDFC Top 100 Fund - Regular Plan - Growth
1,100025,0.037076,0.039155,NaN,HDFC Short Term Debt Fund - Regular - Growth
2,100033,0.532772,0.324340,NaN,HDFC Mid-Cap Opportunities Fund - Regular - Gr...
3,101206,0.479638,0.289602,NaN,ABSL Frontline Equity Fund - Regular - Growth
4,101207,-0.240003,-0.041515,NaN,ABSL Small Cap Fund - Regular - Growth
5,101208,0.072418,0.063143,NaN,ABSL Liquid Fund - Regular - Growth
6,102885,0.202229,0.196624,NaN,UTI Nifty 50 Index Fund - Regular - Growth
7,102886,-0.168080,-0.007672,NaN,UTI Mid Cap Fund - Regular - Growth
8,102887,0.135930,0.255497,NaN,UTI Flexi Cap Fund - Regular - Growth
9,118632,0.340079,0.226466,NaN,Nippon India Large Cap Fund - Regular - Growth


In [24]:
Rf = 0.065  # RBI repo rate proxy, annualized

sharpe_results = []
for code, group in nav_history.groupby("amfi_code"):
    returns = group["daily_return"].dropna()
    if len(returns) < 2:
        continue
    ann_return = returns.mean() * 252
    ann_std = returns.std() * np.sqrt(252)
    sharpe = (ann_return - Rf) / ann_std if ann_std > 0 else np.nan
    sharpe_results.append({"amfi_code": code, "sharpe_ratio": sharpe})

sharpe_df = pd.DataFrame(sharpe_results)
sharpe_df["rank"] = sharpe_df["sharpe_ratio"].rank(ascending=False)
sharpe_df = sharpe_df.sort_values("sharpe_ratio", ascending=False)
sharpe_df

,amfi_code,sharpe_ratio,rank
34,148567,1.068224,1.0
30,120843,0.965561,2.0
36,148569,0.919047,3.0
25,120505,0.883256,4.0
19,119551,0.860977,5.0
38,149323,0.832885,6.0
2,100033,0.808268,7.0
9,118632,0.758851,8.0
16,119094,0.730547,9.0
3,101206,0.717409,10.0


In [25]:
sortino_results = []
for code, group in nav_history.groupby("amfi_code"):
    returns = group["daily_return"].dropna()
    if len(returns) < 2:
        continue
    ann_return = returns.mean() * 252
    downside_returns = returns[returns < 0]
    downside_std = downside_returns.std() * np.sqrt(252) if len(downside_returns) > 1 else np.nan
    sortino = (ann_return - Rf) / downside_std if downside_std and downside_std > 0 else np.nan
    sortino_results.append({"amfi_code": code, "sortino_ratio": sortino})

sortino_df = pd.DataFrame(sortino_results)
sortino_df

,amfi_code,sortino_ratio
0,100016,-0.472822
1,100025,-1.461220
2,100033,1.144216
3,101206,1.063909
4,101207,0.075668
5,101208,-8.741654
6,102885,0.772972
7,102886,-0.420589
8,102887,0.571858
9,118632,1.098880


In [26]:
print(benchmark["index_name"].unique())

<StringArray>
[        'NIFTY50',        'NIFTY100', 'NIFTY_MIDCAP150',    'BSE_SMALLCAP',
        'NIFTY500',   'CRISIL_LIQUID',     'CRISIL_GILT']
Length: 7, dtype: str


In [27]:
# Check benchmark columns first
print(benchmark.columns.tolist())
print(benchmark.head())

['date', 'index_name', 'close_value']
         date index_name  close_value
0  2022-01-03    NIFTY50     17492.79
1  2022-01-04    NIFTY50     17689.64
2  2022-01-05    NIFTY50     17835.05
3  2022-01-06    NIFTY50     17878.51
4  2022-01-07    NIFTY50     17759.15


In [28]:
benchmark["date"] = pd.to_datetime(benchmark["date"])
nifty100 = benchmark[benchmark["index_name"] == "NIFTY100"].copy()  # adjust exact string once confirmed
nifty100 = nifty100.sort_values("date")
nifty100["benchmark_return"] = nifty100["close_value"].pct_change()

alpha_beta_results = []
for code, group in nav_history.groupby("amfi_code"):
    merged = group.merge(nifty100[["date","benchmark_return"]], on="date", how="inner").dropna(
        subset=["daily_return","benchmark_return"])
    if len(merged) < 30:
        continue
    slope, intercept, r_value, p_value, std_err = stats.linregress(
        merged["benchmark_return"], merged["daily_return"])
    beta = slope
    alpha = intercept * 252
    alpha_beta_results.append({"amfi_code": code, "alpha": alpha, "beta": beta, "r_squared": r_value**2})

alpha_beta_df = pd.DataFrame(alpha_beta_results)
alpha_beta_df

,amfi_code,alpha,beta,r_squared
0,100016,0.037476,-0.058268,2.665033e-03
1,100025,0.042818,0.001158,1.461284e-05
2,100033,0.271954,0.005104,1.206652e-05
3,101206,0.213998,0.021086,3.480482e-04
4,101207,0.108971,-0.065289,1.064105e-03
5,101208,0.060861,0.000267,4.607624e-05
6,102885,0.170488,-0.019487,3.828503e-04
7,102886,0.028969,-0.042125,8.964295e-04
8,102887,0.162113,0.016683,1.861917e-04
9,118632,0.218294,-0.008354,5.791756e-05


In [29]:
mdd_results = []
for code, group in nav_history.groupby("amfi_code"):
    group = group.sort_values("date")
    running_max = group["nav"].cummax()
    drawdown = group["nav"] / running_max - 1
    max_dd = drawdown.min()
    worst_idx = drawdown.idxmin()
    worst_date = group.loc[worst_idx, "date"]
    peak_date = group.loc[:worst_idx][group.loc[:worst_idx, "nav"] == running_max.loc[worst_idx]]["date"].iloc[0] if len(group.loc[:worst_idx]) > 0 else None
    mdd_results.append({"amfi_code": code, "max_drawdown": max_dd, "trough_date": worst_date})

mdd_df = pd.DataFrame(mdd_results)
mdd_df

,amfi_code,max_drawdown,trough_date
0,100016,-0.247344,2022-09-15
1,100025,-0.043083,2023-07-28
2,100033,-0.162172,2022-05-12
3,101206,-0.112916,2023-07-05
4,101207,-0.354469,2026-05-11
5,101208,-0.001622,2023-09-12
6,102885,-0.108599,2022-03-29
7,102886,-0.280011,2026-04-27
8,102887,-0.215398,2022-07-04
9,118632,-0.174141,2024-07-19


In [30]:
scorecard = cagr_df[["amfi_code","scheme_name","cagr_3yr"]].merge(
    sharpe_df[["amfi_code","sharpe_ratio"]], on="amfi_code").merge(
    alpha_beta_df[["amfi_code","alpha"]], on="amfi_code").merge(
    fund_master[["amfi_code","expense_ratio_pct"]], on="amfi_code").merge(
    mdd_df[["amfi_code","max_drawdown"]], on="amfi_code")

def rank_to_score(series, ascending=False):
    ranks = series.rank(ascending=ascending, pct=True)
    return ranks * 100

scorecard["return_score"] = rank_to_score(scorecard["cagr_3yr"])
scorecard["sharpe_score"] = rank_to_score(scorecard["sharpe_ratio"])
scorecard["alpha_score"] = rank_to_score(scorecard["alpha"])
scorecard["expense_score"] = rank_to_score(scorecard["expense_ratio_pct"], ascending=True)  # lower is better
scorecard["dd_score"] = rank_to_score(scorecard["max_drawdown"], ascending=True)  # closer to 0 (less negative) is better

scorecard["overall_score"] = (
    0.30 * scorecard["return_score"] +
    0.25 * scorecard["sharpe_score"] +
    0.20 * scorecard["alpha_score"] +
    0.15 * scorecard["expense_score"] +
    0.10 * scorecard["dd_score"]
)

scorecard = scorecard.sort_values("overall_score", ascending=False)
scorecard.to_csv("../fund_scorecard.csv", index=False)
scorecard

,amfi_code,scheme_name,cagr_3yr,sharpe_ratio,alpha,expense_ratio_pct,max_drawdown,return_score,sharpe_score,alpha_score,expense_score,dd_score,overall_score
14,119092,Axis Bluechip Fund - Regular - Growth,0.005258,-0.131009,0.068995,1.64,-0.144016,90.0,75.0,75.0,98.75,65.0,82.0625
0,100016,HDFC Top 100 Fund - Regular Plan - Growth,0.012924,-0.321019,0.037476,1.55,-0.247344,87.5,85.0,97.5,80.00,17.5,80.7500
7,102886,UTI Mid Cap Fund - Regular - Growth,-0.007672,-0.294889,0.028969,1.51,-0.280011,92.5,82.5,100.0,61.25,15.0,79.0625
5,101208,ABSL Liquid Fund - Regular - Growth,0.063143,-4.650401,0.060861,0.79,-0.001622,77.5,100.0,82.5,27.50,95.0,78.3750
1,100025,HDFC Short Term Debt Fund - Regular - Growth,0.039155,-1.039941,0.042818,0.56,-0.043083,85.0,92.5,95.0,5.00,92.5,77.6250
17,119095,Axis Small Cap Fund - Regular - Growth,-0.117033,-0.151658,0.048016,1.38,-0.516778,100.0,80.0,92.5,42.50,5.0,75.3750
18,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,0.058390,-0.743099,0.056209,0.77,-0.043287,80.0,87.5,85.0,22.50,90.0,75.2500
13,118636,Nippon India Gilt Securities Fund - Regular - ...,0.040613,-0.850579,0.050748,0.55,-0.083164,82.5,90.0,87.5,2.50,87.5,73.8750
31,120844,Kotak Liquid Fund - Regular - Growth,0.066944,-4.000349,0.064557,0.60,-0.001163,75.0,97.5,80.0,7.50,97.5,73.7500
27,120507,ICICI Pru Liquid Fund - Regular - Growth,0.073925,-3.650184,0.067462,0.74,-0.000977,72.5,95.0,77.5,17.50,100.0,73.6250


In [33]:
nifty100 = benchmark[benchmark["index_name"] == "NIFTY100"].copy()
nifty100 = nifty100.sort_values("date")
nifty100["benchmark_return"] = nifty100["close_value"].pct_change()
nifty100["cum_return"] = (1 + nifty100["benchmark_return"].fillna(0)).cumprod() - 1

nifty50 = benchmark[benchmark["index_name"] == "NIFTY50"].copy()
nifty50 = nifty50.sort_values("date")
nifty50["cum_return"] = (1 + nifty50["close_value"].pct_change().fillna(0)).cumprod() - 1

top5_codes = scorecard.head(5)["amfi_code"].tolist()

fig = go.Figure()
for code in top5_codes:
    fund_data = nav_history[nav_history["amfi_code"] == code].sort_values("date").tail(756)
    fund_data = fund_data.copy()
    fund_data["cum_return"] = (fund_data["nav"] / fund_data["nav"].iloc[0]) - 1
    name = fund_master[fund_master["amfi_code"]==code]["scheme_name"].values[0]
    fig.add_trace(go.Scatter(x=fund_data["date"], y=fund_data["cum_return"], name=name))

fig.add_trace(go.Scatter(x=nifty50["date"], y=nifty50["cum_return"], name="Nifty 50", line=dict(dash="dash")))
fig.add_trace(go.Scatter(x=nifty100["date"], y=nifty100["cum_return"], name="Nifty 100", line=dict(dash="dash")))
fig.update_layout(title="Top 5 Funds vs Nifty 50 & Nifty 100 (3yr Cumulative Return)")
fig.write_image(CHARTS + "10_benchmark_comparison.png", width=1400, height=700)
fig.show()

# Tracking error
for code in top5_codes:
    merged = nav_history[nav_history["amfi_code"]==code].merge(
        nifty100[["date","benchmark_return"]], on="date", how="inner").dropna(
        subset=["daily_return","benchmark_return"])
    te = (merged["daily_return"] - merged["benchmark_return"]).std() * np.sqrt(252)
    print(f"{code}: tracking error = {te:.4f}")

119092: tracking error = 0.1891
100016: tracking error = 0.1993
102886: tracking error = 0.2256
101208: tracking error = 0.1290
100025: tracking error = 0.1345


In [32]:
scorecard.to_csv("../fund_scorecard.csv", index=False)
alpha_beta_df.to_csv("../alpha_beta.csv", index=False)